In [0]:
# ============================================================
# MVP VTI CBL
# Notebook: 03_Gold
#
# Objetivo:
# Relacionar cada medição CBL ao:
# - Local de instalação
# - Bloco ferroviário de 25 metros
#
# Regra espacial:
# mesma linha
# 
# inicio_m <= posicao_m < fim_m
#
# Saída principal:
# mvp_vti.gold.fato_cbl_bloco
# ============================================================

In [0]:
# Importa as funções do PySpark utilizadas nos relacionamentos, validações e agregações da camada Gold
from pyspark.sql import functions as F
# Importa Window para operações analíticas e de ordenação utilizadas ao longo do desenvolvimento
from pyspark.sql.window import Window

In [0]:
# Carrega as bases tratadas na Silver que serão utilizadas na integração espacial da camada Gold
df_cbl = spark.table("mvp_vti.silver.silver_cbl")
df_locais = spark.table("mvp_vti.silver.silver_locais")
df_blocos = spark.table("mvp_vti.silver.silver_blocos")


In [0]:
# Relaciona cada medição CBL ao Local de Instalação correspondente
# A associação considera a mesma linha ferroviária e a posição da medição dentro do intervalo do local
# O left join preserva também os registros sem correspondência para posterior avaliação da cobertura
df_cbl_local = (
df_cbl.alias("cbl")
.join(
df_locais.alias("loc"),
(
(F.col("cbl.linha") == F.col("loc.linha"))
& (F.col("cbl.posicao_m") >= F.col("loc.inicio_m"))
& (F.col("cbl.posicao_m") < F.col("loc.fim_m"))
),
"left"
)
)

In [0]:
# Exibe uma amostra do relacionamento entre as medições CBL e os Locais de Instalação
# A visualização permite conferir a linha, a posição da medição e o intervalo espacial associado
display(
df_cbl_local.select(
F.col("cbl.id"),
F.col("cbl.linha"),
F.col("cbl.posicao_m"),
F.col("cbl.date_time"),
F.col("cbl.cbl"),
F.col("loc.taxonomia"),
F.col("loc.inicio_m").alias("local_inicio_m"),
F.col("loc.fim_m").alias("local_fim_m")
).limit(50)
)

id,linha,posicao_m,date_time,cbl,taxonomia,local_inicio_m,local_fim_m
26,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,null,null
32,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.14,null,null,null
34,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,null
40,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,null
46,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.17,null,null,null
47,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,null,null
53,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,null
89,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.11,null,null,null
98,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.12,null,null,null
105,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.13,null,null,null


In [0]:
# Calcula o total de registros resultantes do relacionamento entre CBL e Locais
total = df_cbl_local.count()
# Conta as medições que encontraram um Local de Instalação correspondente
com_local = (
df_cbl_local
.filter(F.col("loc.taxonomia").isNotNull())
.count()
)
# Conta as medições que permaneceram sem correspondência na base de Locais
sem_local = (
df_cbl_local
.filter(F.col("loc.taxonomia").isNull())
.count()
)
# Calcula e apresenta a cobertura obtida no relacionamento espacial
cobertura_local = round(com_local / total * 100, 2)
print("=== COBERTURA CBL × LOCAIS ===")
print("Total:", total)
print("Com local:", com_local)
print("Sem local:", sem_local)
print("Cobertura:", cobertura_local, "%")

=== COBERTURA CBL × LOCAIS ===
Total: 42881
Com local: 39546
Sem local: 3335
Cobertura: 92.22 %


In [0]:
# Relaciona as medições CBL aos Blocos de 25 m utilizando a mesma referência espacial
# A associação considera a linha ferroviária e a posição da medição dentro dos limites de cada bloco
# O left join mantém também os registros sem bloco para permitir a avaliação posterior da cobertura
df_gold = (
df_cbl_local.alias("base")
.join(
df_blocos.alias("bloco"),
(
(F.col("base.cbl.linha") == F.col("bloco.linha"))
& (F.col("base.cbl.posicao_m") >= F.col("bloco.inicio_m"))
& (F.col("base.cbl.posicao_m") < F.col("bloco.fim_m"))
),
"left"
)
)

In [0]:
# Organiza o resultado do relacionamento CBL × Locais em uma estrutura mais simples
# Mantém os campos necessários para a próxima etapa de integração com os Blocos de 25 m
df_cbl_local_limpo = (
df_cbl_local
.select(
F.col("cbl.id").alias("id"),
F.col("cbl.linha").alias("linha"),
F.col("cbl.posicao_m").alias("posicao_m"),
F.col("cbl.date_time").alias("date_time"),
F.col("cbl.cbl").alias("cbl"),
F.col("cbl.km_major").alias("km_major"),
F.col("cbl.kp_minor").alias("kp_minor"),
F.col("cbl.track_no").alias("track_no"),
F.col("cbl.speed").alias("speed"),
F.col("cbl.direction").alias("direction"),
F.col("loc.taxonomia").alias("taxonomia"),
F.col("loc.inicio_m").alias("local_inicio_m"),
F.col("loc.fim_m").alias("local_fim_m")
)
# Identifica as medições válidas de CBL que não encontraram correspondência na base de Locais
.withColumn("sem_local", F.col("taxonomia").isNull())
)
# Exibe uma amostra para conferir a estrutura consolidada após o primeiro relacionamento espacial
display(df_cbl_local_limpo.limit(50))
# Confere a distribuição dos registros com e sem Local de Instalação associado
display(
df_cbl_local_limpo
.groupBy("sem_local")
.count()
)

id,linha,posicao_m,date_time,cbl,km_major,kp_minor,track_no,speed,direction,taxonomia,local_inicio_m,local_fim_m,sem_local
26,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,null,UNKNOWN,24.0,N/A,null,null,null,true
32,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.14,null,null,UNKNOWN,27.0,N/A,null,null,null,true
34,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,UNKNOWN,25.0,N/A,null,null,null,true
40,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,UNKNOWN,27.0,N/A,null,null,null,true
46,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.17,null,null,UNKNOWN,25.0,N/A,null,null,null,true
47,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,null,UNKNOWN,25.0,N/A,null,null,null,true
53,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,null,UNKNOWN,28.0,N/A,null,null,null,true
89,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.11,null,null,UNKNOWN,24.0,N/A,null,null,null,true
98,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.12,null,null,UNKNOWN,27.0,N/A,null,null,null,true
105,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.13,null,null,UNKNOWN,28.0,N/A,null,null,null,true


sem_local,count
true,3335
false,39546


In [0]:
# Relaciona cada medição CBL ao Bloco de 25 m correspondente na base oficial de blocos
# A associação considera a mesma linha ferroviária e a posição da medição dentro do intervalo do bloco
# O left join preserva também as medições sem correspondência para posterior avaliação da cobertura
df_cbl_local_bloco = (
df_cbl_local_limpo.alias("base")
.join(
df_blocos.alias("bloco"),
(
(F.col("base.linha") == F.col("bloco.linha"))
& (F.col("base.posicao_m") >= F.col("bloco.inicio_m"))
& (F.col("base.posicao_m") < F.col("bloco.fim_m"))
),
"left"
)
)

In [0]:
# Exibe uma amostra do resultado após a integração das medições CBL com Locais e Blocos de 25 m
# A visualização permite conferir a posição da medição e os respectivos intervalos espaciais associados
display(
df_cbl_local_bloco.select(
F.col("base.id").alias("id"),
F.col("base.linha").alias("linha"),
F.col("base.posicao_m").alias("posicao_m"),
F.col("base.date_time").alias("date_time"),
F.col("base.cbl").alias("cbl"),
F.col("base.taxonomia").alias("taxonomia"),
F.col("base.sem_local").alias("sem_local"),
F.col("bloco.bloco_id").alias("bloco_id"),
F.col("bloco.inicio_m").alias("bloco_inicio_m"),
F.col("bloco.fim_m").alias("bloco_fim_m")
).limit(50)
)

id,linha,posicao_m,date_time,cbl,taxonomia,sem_local,bloco_id,bloco_inicio_m,bloco_fim_m
26,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,true,null,null,null
32,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.14,null,true,null,null,null
34,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,true,null,null,null
40,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,true,null,null,null
46,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.17,null,true,null,null,null
47,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.12,null,true,null,null,null
53,UNKNOWN,null,2025-03-15T11:55:00.000Z,0.11,null,true,null,null,null
89,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.11,null,true,null,null,null
98,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.12,null,true,null,null,null
105,UNKNOWN,null,2025-03-15T13:09:00.000Z,0.13,null,true,null,null,null


In [0]:
# Considera como referência o total de medições CBL disponíveis após a integração com Locais
total_cbl = df_cbl_local_limpo.count()
# Conta as medições que encontraram correspondência em um Bloco de 25 m
com_bloco = (
df_cbl_local_bloco
.filter(F.col("bloco.bloco_id").isNotNull())
.select(F.col("base.id"))
.distinct()
.count()
)
# Conta as medições que permaneceram sem correspondência na base oficial de Blocos
sem_bloco = (
df_cbl_local_bloco
.filter(F.col("bloco.bloco_id").isNull())
.select(F.col("base.id"))
.distinct()
.count()
)
# Calcula a cobertura obtida no relacionamento espacial CBL × Blocos
cobertura_bloco = com_bloco / total_cbl * 100
print("=== COBERTURA CBL × BLOCOS 25M ===")
print(f"Total CBL: {total_cbl}")
print(f"Com bloco: {com_bloco}")
print(f"Sem bloco: {sem_bloco}")
print(f"Cobertura: {cobertura_bloco:.2f}%")
# Verifica se uma mesma medição CBL foi associada a mais de um bloco
# O resultado esperado é zero, indicando ausência de ambiguidade no relacionamento espacial
duplicidade_bloco = (
df_cbl_local_bloco
.filter(F.col("bloco.bloco_id").isNotNull())
.groupBy(F.col("base.id").alias("id"))
.agg(
F.countDistinct(F.col("bloco.bloco_id")).alias("qtd_blocos")
)
.filter(F.col("qtd_blocos") > 1)
)
print(
"Medições associadas a mais de um bloco:",
duplicidade_bloco.count()
)

=== COBERTURA CBL × BLOCOS 25M ===
Total CBL: 42881
Com bloco: 41499
Sem bloco: 1382
Cobertura: 96.78%
Medições associadas a mais de um bloco: 0


In [0]:
# Consolida os dados da medição CBL com as informações espaciais de Local e Bloco de 25 m
# Esta estrutura representa a base analítica principal da camada Gold
df_fato_cbl_bloco = (
df_cbl_local_bloco
.select(
# Identificação e momento da medição
F.col("base.id").alias("id"),
F.col("base.date_time").alias("date_time"),
# Referência ferroviária da medição
F.col("base.linha").alias("linha"),
F.col("base.km_major").alias("km_major"),
F.col("base.kp_minor").alias("kp_minor"),
F.col("base.posicao_m").alias("posicao_m"),
# Valor CBL
F.col("base.cbl").alias("cbl"),
# Informações operacionais disponíveis na fonte VTI
F.col("base.track_no").alias("track_no"),
F.col("base.speed").alias("speed"),
F.col("base.direction").alias("direction"),
# Local de Instalação associado à posição da medição
F.col("base.taxonomia").alias("taxonomia"),
F.col("base.local_inicio_m").alias("local_inicio_m"),
F.col("base.local_fim_m").alias("local_fim_m"),
F.col("base.sem_local").alias("sem_local"),
# Bloco oficial de 25 m associado à posição da medição
F.col("bloco.bloco_id").alias("bloco_id"),
F.col("bloco.inicio_m").alias("bloco_inicio_m"),
F.col("bloco.fim_m").alias("bloco_fim_m")
)
# Identifica as medições que não encontraram correspondência na base oficial de Blocos
.withColumn("sem_bloco", F.col("bloco_id").isNull())
)
# Exibe uma amostra para conferir a estrutura final antes da persistência da tabela fato
display(df_fato_cbl_bloco.limit(1500))

id,date_time,linha,km_major,kp_minor,posicao_m,cbl,track_no,speed,direction,taxonomia,local_inicio_m,local_fim_m,sem_local,bloco_id,bloco_inicio_m,bloco_fim_m,sem_bloco
26,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.12,UNKNOWN,24.0,N/A,null,null,null,true,null,null,null,true
32,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.14,UNKNOWN,27.0,N/A,null,null,null,true,null,null,null,true
34,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.11,UNKNOWN,25.0,N/A,null,null,null,true,null,null,null,true
40,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.11,UNKNOWN,27.0,N/A,null,null,null,true,null,null,null,true
46,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.17,UNKNOWN,25.0,N/A,null,null,null,true,null,null,null,true
47,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.12,UNKNOWN,25.0,N/A,null,null,null,true,null,null,null,true
53,2025-03-15T11:55:00.000Z,UNKNOWN,null,null,null,0.11,UNKNOWN,28.0,N/A,null,null,null,true,null,null,null,true
89,2025-03-15T13:09:00.000Z,UNKNOWN,null,null,null,0.11,UNKNOWN,24.0,N/A,null,null,null,true,null,null,null,true
98,2025-03-15T13:09:00.000Z,UNKNOWN,null,null,null,0.12,UNKNOWN,27.0,N/A,null,null,null,true,null,null,null,true
105,2025-03-15T13:09:00.000Z,UNKNOWN,null,null,null,0.13,UNKNOWN,28.0,N/A,null,null,null,true,null,null,null,true


In [0]:
# Considera o total de medições disponíveis na tabela fato construída
total = df_fato_cbl_bloco.count()
# Identifica as medições que possuem simultaneamente Local de Instalação e Bloco de 25 m associados
completo = (
df_fato_cbl_bloco
.filter(
(~F.col("sem_local"))
& (~F.col("sem_bloco"))
)
.count()
)
# Calcula a cobertura completa da integração espacial das três fontes
cobertura_completa = completo / total * 100
print("=== COBERTURA ESPACIAL COMPLETA ===")
print(f"Total CBL: {total}")
print(f"Com Local + Bloco: {completo}")
print(f"Cobertura completa: {cobertura_completa:.2f}%")

=== COBERTURA ESPACIAL COMPLETA ===
Total CBL: 42881
Com Local + Bloco: 39546
Cobertura completa: 92.22%


In [0]:
# Persiste a tabela fato como produto de dados principal da camada Gold
# O overwriteSchema mantém o schema da tabela alinhado à estrutura construída no pipeline
(
df_fato_cbl_bloco.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("mvp_vti.gold.fato_cbl_bloco")
)
# Recarrega a tabela diretamente do Catalog para confirmar que a persistência foi concluída
df_gold_validacao = spark.table("mvp_vti.gold.fato_cbl_bloco")
# Confere a quantidade de registros disponíveis na tabela Gold persistida
print("Registros Gold:", df_gold_validacao.count())
# Exibe uma amostra do produto de dados final para validar sua estrutura e conteúdo
display(df_gold_validacao.limit(1500))

Registros Gold: 42881


id,date_time,linha,km_major,kp_minor,posicao_m,cbl,track_no,speed,direction,taxonomia,local_inicio_m,local_fim_m,sem_local,bloco_id,bloco_inicio_m,bloco_fim_m,sem_bloco
6028,2025-03-16T21:01:00.000Z,LP002,819,472,819472,0.17,2,75.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819450_819475,819450,819475,false
12529,2025-03-23T07:12:00.000Z,LP002,819,444,819444,0.16,2,67.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819425_819450,819425,819450,false
18046,2025-03-26T15:32:00.000Z,LP002,819,447,819447,0.15,2,75.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819425_819450,819425,819450,false
26182,2025-03-31T07:35:00.000Z,LP002,819,442,819442,0.15,2,74.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819425_819450,819425,819450,false
37256,2025-04-14T02:11:00.000Z,LP002,819,437,819437,0.15,2,65.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819425_819450,819425,819450,false
6030,2025-03-16T21:01:00.000Z,LP002,819,418,819418,0.13,2,75.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819400_819425,819400,819425,false
37313,2025-04-14T02:11:00.000Z,LP002,819,378,819378,0.11,2,65.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819375_819400,819375,819400,false
37296,2025-04-14T02:11:00.000Z,LP002,819,303,819303,0.13,2,65.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819300_819325,819300,819325,false
38054,2025-04-14T17:45:00.000Z,LP002,819,300,819300,0.11,2,51.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819300_819325,819300,819325,false
6040,2025-03-16T21:01:00.000Z,LP002,819,115,819115,0.21,2,75.0,N/A,SEG-VF - TU00819-L0002-CDVW02T,819050,819504,false,LP002_819100_819125,819100,819125,false


In [0]:
# Carrega a tabela fato já persistida no Catalog para iniciar as agregações analíticas da camada Gold
df_fato = spark.table("mvp_vti.gold.fato_cbl_bloco")

In [0]:
# Cria a coluna data a partir de date_time para permitir as agregações diárias da camada Gold
df_fato = (
df_fato
.withColumn("data", F.to_date("date_time"))
)
# Exibe uma amostra para validar a preparação da granularidade diária utilizada nas etapas seguintes
display(
df_fato.select(
"date_time",
"data",
"linha",
"bloco_id",
"cbl"
).limit(20)
)

date_time,data,linha,bloco_id,cbl
2025-03-16T21:01:00.000Z,2025-03-16,LP002,LP002_819450_819475,0.17
2025-03-23T07:12:00.000Z,2025-03-23,LP002,LP002_819425_819450,0.16
2025-03-26T15:32:00.000Z,2025-03-26,LP002,LP002_819425_819450,0.15
2025-03-31T07:35:00.000Z,2025-03-31,LP002,LP002_819425_819450,0.15
2025-04-14T02:11:00.000Z,2025-04-14,LP002,LP002_819425_819450,0.15
2025-03-16T21:01:00.000Z,2025-03-16,LP002,LP002_819400_819425,0.13
2025-04-14T02:11:00.000Z,2025-04-14,LP002,LP002_819375_819400,0.11
2025-04-14T02:11:00.000Z,2025-04-14,LP002,LP002_819300_819325,0.13
2025-04-14T17:45:00.000Z,2025-04-14,LP002,LP002_819300_819325,0.11
2025-03-16T21:01:00.000Z,2025-03-16,LP002,LP002_819100_819125,0.21


In [0]:
# Seleciona somente as medições que possuem um Bloco de 25 m associado
# Os registros sem bloco permanecem preservados na tabela fato, mas não entram nas agregações por bloco
df_fato_com_bloco = (
df_fato
.filter(F.col("bloco_id").isNotNull())
)
# Confere o volume total da tabela fato e a quantidade disponível para as análises por bloco
print("Total fato:", df_fato.count())
print("Com bloco:", df_fato_com_bloco.count())

Total fato: 42881
Com bloco: 41499


In [0]:
# Consolida as medições CBL por dia, linha e Bloco de 25 m
# A agregação cria indicadores que permitem acompanhar o comportamento do CBL ao longo do tempo
df_agg_bloco_dia = (
df_fato_com_bloco
.groupBy(
"data",
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m"
)
.agg(
# Quantidade de medições disponíveis no bloco durante o dia
F.count("*").alias("qtd_medicoes"),
# Indicadores estatísticos do CBL no período
F.avg("cbl").alias("cbl_medio"),
F.max("cbl").alias("cbl_max"),
F.min("cbl").alias("cbl_min"),
F.expr("percentile_approx(cbl, 0.50)").alias("cbl_p50"),
F.expr("percentile_approx(cbl, 0.95)").alias("cbl_p95"),
F.stddev("cbl").alias("cbl_std")
)
)

In [0]:
# Exibe uma amostra da agregação diária ordenada por data, linha e posição inicial do bloco
# A visualização permite conferir os indicadores calculados para cada Bloco de 25 m ao longo do tempo
display(
df_agg_bloco_dia
.orderBy(
"data",
"linha",
"bloco_inicio_m"
)
.limit(50)
)

data,linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_min,cbl_p50,cbl_p95,cbl_std
2025-03-15,LP001,LP001_139050_139075,139050,139075,1,0.11,0.11,0.11,0.11,0.11,null
2025-03-15,LP001,LP001_139075_139100,139075,139100,1,0.15,0.15,0.15,0.15,0.15,null
2025-03-15,LP001,LP001_139100_139125,139100,139125,1,0.13,0.13,0.13,0.13,0.13,null
2025-03-15,LP001,LP001_139175_139200,139175,139200,1,0.15,0.15,0.15,0.15,0.15,null
2025-03-15,LP001,LP001_139225_139250,139225,139250,1,0.13,0.13,0.13,0.13,0.13,null
2025-03-15,LP001,LP001_143175_143200,143175,143200,1,0.11,0.11,0.11,0.11,0.11,null
2025-03-15,LP001,LP001_143925_143950,143925,143950,1,0.13,0.13,0.13,0.13,0.13,null
2025-03-15,LP001,LP001_145875_145900,145875,145900,1,0.16,0.16,0.16,0.16,0.16,null
2025-03-15,LP001,LP001_145975_146000,145975,146000,1,0.15,0.15,0.15,0.15,0.15,null
2025-03-15,LP002,LP002_7325_7350,7325,7350,1,0.18,0.18,0.18,0.18,0.18,null


In [0]:
# Exibe os blocos e dias com maiores valores de CBL P95
# Esta ordenação permite uma primeira leitura dos segmentos com valores relativos mais elevados
display(
df_agg_bloco_dia
.orderBy(F.desc("cbl_p95"))
.limit(20)
)
# Confere o volume total de registros gerados pela agregação diária por bloco
print(
"Registros da agregação:",
df_agg_bloco_dia.count()
)

data,linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_min,cbl_p50,cbl_p95,cbl_std
2026-04-18,LP002,LP002_858175_858200,858175,858200,1,0.41,0.41,0.41,0.41,0.41,null
2026-05-07,LP002,LP002_858175_858200,858175,858200,1,0.4,0.4,0.4,0.4,0.4,null
2026-03-07,LP002,LP002_296075_296100,296075,296100,2,0.295,0.36,0.23,0.23,0.36,0.09192388155425116
2025-09-19,LP002,LP002_809450_809475,809450,809475,1,0.35,0.35,0.35,0.35,0.35,null
2026-04-26,LP002,LP002_858175_858200,858175,858200,1,0.35,0.35,0.35,0.35,0.35,null
2026-04-18,LP002,LP002_858325_858350,858325,858350,1,0.33,0.33,0.33,0.33,0.33,null
2025-09-06,LP002,LP002_809450_809475,809450,809475,1,0.33,0.33,0.33,0.33,0.33,null
2026-01-11,LP002,LP002_858175_858200,858175,858200,1,0.33,0.33,0.33,0.33,0.33,null
2025-04-23,LP002,LP002_858175_858200,858175,858200,1,0.33,0.33,0.33,0.33,0.33,null
2025-09-02,LP002,LP002_858175_858200,858175,858200,1,0.33,0.33,0.33,0.33,0.33,null


Registros da agregação: 40577


In [0]:
# Persiste a agregação diária de CBL por Bloco de 25 m como tabela Delta na camada Gold
# O overwriteSchema mantém a estrutura da tabela alinhada aos indicadores calculados nesta etapa
(
df_agg_bloco_dia.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("mvp_vti.gold.agg_cbl_bloco_dia")
)

In [0]:
# Recarrega a agregação diária diretamente do Catalog para validar a persistência da tabela Gold
df_agg_validacao = spark.table("mvp_vti.gold.agg_cbl_bloco_dia")
# Confere a quantidade de registros disponíveis após a gravação
print("Registros:", df_agg_validacao.count())
# Exibe uma amostra da tabela persistida para validar sua estrutura e os indicadores calculados
display(df_agg_validacao.limit(50))

Registros: 40577


data,linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_min,cbl_p50,cbl_p95,cbl_std
2025-04-18,LP001,LP001_807200_807225,807200,807225,1,0.12,0.12,0.12,0.12,0.12,null
2025-03-27,LP001,LP001_805150_805175,805150,805175,1,0.14,0.14,0.14,0.14,0.14,null
2025-04-01,LP001,LP001_805150_805175,805150,805175,1,0.13,0.13,0.13,0.13,0.13,null
2025-04-14,LP001,LP001_804975_805000,804975,805000,1,0.14,0.14,0.14,0.14,0.14,null
2025-03-27,LP002,LP002_810575_810600,810575,810600,1,0.11,0.11,0.11,0.11,0.11,null
2025-04-14,LP002,LP002_809725_809750,809725,809750,1,0.12,0.12,0.12,0.12,0.12,null
2025-04-14,LP002,LP002_809700_809725,809700,809725,1,0.11,0.11,0.11,0.11,0.11,null
2025-04-06,LP002,LP002_809500_809525,809500,809525,1,0.11,0.11,0.11,0.11,0.11,null
2025-03-27,LP001,LP001_795050_795075,795050,795075,1,0.12,0.12,0.12,0.12,0.12,null
2025-03-27,LP002,LP002_805825_805850,805825,805850,1,0.13,0.13,0.13,0.13,0.13,null


In [0]:
# Consolida todas as medições disponíveis para cada Bloco de 25 m ao longo do período analisado
# Esta visão permite comparar os blocos utilizando indicadores estatísticos do CBL
df_agg_bloco = (
df_fato_com_bloco
.groupBy(
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m"
)
.agg(
# Quantidade de medições disponíveis para cada bloco
F.count("*").alias("qtd_medicoes"),
# Indicadores utilizados para caracterizar o comportamento do CBL no período
F.avg("cbl").alias("cbl_medio"),
F.max("cbl").alias("cbl_max"),
F.expr("percentile_approx(cbl, 0.95)").alias("cbl_p95"),
F.stddev("cbl").alias("cbl_std"),
# Limites temporais das medições disponíveis para cada bloco
F.min("date_time").alias("primeira_medicao"),
F.max("date_time").alias("ultima_medicao")
)
)
# Exibe os 20 blocos com maiores valores de CBL P95 para uma primeira avaliação da criticidade relativa
display(
df_agg_bloco
.orderBy(F.desc("cbl_p95"))
.limit(20)
)

linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_p95,cbl_std,primeira_medicao,ultima_medicao
LP002,LP002_858175_858200,858175,858200,62,0.22322580645161294,0.41,0.33,0.06367886113849987,2025-03-16T21:33:00.000Z,2026-05-28T12:03:00.000Z
LP002,LP002_809450_809475,809450,809475,38,0.15184210526315792,0.35,0.33,0.06194397046145139,2025-04-21T06:51:00.000Z,2026-05-11T19:52:00.000Z
LP002,LP002_551950_551975,551950,551975,24,0.16,0.32,0.32,0.06587207629020517,2025-04-13T19:18:00.000Z,2026-05-10T16:50:00.000Z
LP002,LP002_323625_323650,323625,323650,26,0.2034615384615384,0.31,0.31,0.05775412073210414,2025-08-27T06:38:00.000Z,2026-05-09T05:52:00.000Z
LP002,LP002_114050_114075,114050,114075,15,0.14666666666666667,0.31,0.31,0.056146068004633704,2025-03-15T21:49:00.000Z,2026-06-10T04:54:00.000Z
LP001,LP001_145900_145925,145900,145925,2,0.21,0.3,0.3,0.12727922061357855,2025-12-29T03:12:00.000Z,2026-03-07T01:40:00.000Z
LP002,LP002_261800_261825,261800,261825,12,0.1975,0.3,0.3,0.07212678231609064,2025-03-28T12:54:00.000Z,2026-06-10T11:17:00.000Z
LP002,LP002_338975_339000,338975,339000,17,0.16117647058823525,0.3,0.3,0.05743282521141291,2025-08-30T23:53:00.000Z,2026-06-07T13:00:00.000Z
LP001,LP001_830250_830275,830250,830275,4,0.1775,0.3,0.3,0.08381527307120104,2025-09-06T03:04:00.000Z,2026-04-27T14:45:00.000Z
LP001,LP001_244850_244875,244850,244875,15,0.1753333333333333,0.3,0.3,0.055916604570258606,2025-03-30T16:58:00.000Z,2026-05-06T18:11:00.000Z


In [0]:
# Persiste a visão consolidada de CBL por Bloco de 25 m como tabela Delta na camada Gold
# Esta tabela será utilizada posteriormente no ranking de criticidade relativa dos segmentos
(
df_agg_bloco.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("mvp_vti.gold.agg_cbl_bloco")
)

In [0]:
# Agrupa as medições por semana para reduzir oscilações diárias e facilitar a análise temporal por bloco
df_agg_bloco_semana = (
df_fato_com_bloco
.withColumn(
"semana",
F.date_trunc("week", F.col("date_time"))
)
.groupBy(
"semana",
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m"
)
.agg(
# Quantidade de medições disponíveis para o bloco em cada semana
F.count("*").alias("qtd_medicoes"),
# Indicadores utilizados para acompanhar o comportamento semanal do CBL
F.avg("cbl").alias("cbl_medio"),
F.max("cbl").alias("cbl_max"),
F.expr("percentile_approx(cbl, 0.95)").alias("cbl_p95"),
F.stddev("cbl").alias("cbl_std")
)
)
# Exibe uma amostra ordenada para conferir a evolução semanal de cada bloco
display(
df_agg_bloco_semana
.orderBy(
"linha",
"bloco_id",
"semana"
)
.limit(50)
)

semana,linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_p95,cbl_std
2025-10-13T00:00:00.000Z,LP001,LP001_103650_103675,103650,103675,1,0.15,0.15,0.15,null
2025-10-13T00:00:00.000Z,LP001,LP001_103925_103950,103925,103950,1,0.11,0.11,0.11,null
2025-10-13T00:00:00.000Z,LP001,LP001_105375_105400,105375,105400,1,0.11,0.11,0.11,null
2025-10-06T00:00:00.000Z,LP001,LP001_105400_105425,105400,105425,1,0.15,0.15,0.15,null
2026-03-09T00:00:00.000Z,LP001,LP001_105400_105425,105400,105425,1,0.12,0.12,0.12,null
2025-10-13T00:00:00.000Z,LP001,LP001_105425_105450,105425,105450,1,0.13,0.13,0.13,null
2026-05-11T00:00:00.000Z,LP001,LP001_105450_105475,105450,105475,1,0.17,0.17,0.17,null
2025-10-13T00:00:00.000Z,LP001,LP001_105475_105500,105475,105500,1,0.11,0.11,0.11,null
2026-01-26T00:00:00.000Z,LP001,LP001_105525_105550,105525,105550,1,0.12,0.12,0.12,null
2026-05-11T00:00:00.000Z,LP001,LP001_105525_105550,105525,105550,1,0.14,0.14,0.14,null


In [0]:
# Persiste a agregação semanal de CBL por Bloco de 25 m como tabela Delta na camada Gold
# Esta visão será utilizada como base para o cálculo da tendência temporal de cada segmento
(
df_agg_bloco_semana.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("mvp_vti.gold.agg_cbl_bloco_semana")
)
# Confere a quantidade de registros disponíveis após a persistência da agregação semanal
print(
"Registros semanais:",
spark.table("mvp_vti.gold.agg_cbl_bloco_semana").count()
)

Registros semanais: 35369


In [0]:
# Consolida a cobertura temporal disponível para cada Bloco de 25 m
# A quantidade de semanas observadas será utilizada para avaliar se existe histórico suficiente para análise de tendência
df_cobertura_temporal = (
df_agg_bloco_semana
.groupBy(
"linha",
"bloco_id"
)
.agg(
F.countDistinct("semana").alias("qtd_semanas"),
F.min("semana").alias("primeira_semana"),
F.max("semana").alias("ultima_semana")
)
)
# Exibe os blocos com maior quantidade de semanas disponíveis no histórico
display(
df_cobertura_temporal
.orderBy(F.desc("qtd_semanas"))
.limit(50)
)
# Mostra a distribuição da quantidade de semanas disponíveis entre os blocos
display(
df_cobertura_temporal
.groupBy("qtd_semanas")
.count()
.orderBy("qtd_semanas")
)

linha,bloco_id,qtd_semanas,primeira_semana,ultima_semana
LP002,LP002_858175_858200,37,2025-03-10T00:00:00.000Z,2026-05-25T00:00:00.000Z
LP002,LP002_53600_53625,37,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_830325_830350,36,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_830300_830325,34,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_858325_858350,33,2025-03-10T00:00:00.000Z,2026-05-25T00:00:00.000Z
LP002,LP002_551675_551700,33,2025-04-14T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_346950_346975,32,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_764700_764725,32,2025-03-17T00:00:00.000Z,2026-06-01T00:00:00.000Z
LP002,LP002_347175_347200,31,2025-04-07T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_53800_53825,31,2025-03-24T00:00:00.000Z,2026-05-25T00:00:00.000Z


qtd_semanas,count
1,2844
2,1158
3,735
4,529
5,372
6,345
7,243
8,219
9,179
10,152


In [0]:
# Converte a referência semanal em uma variável numérica contínua
# Esta transformação permite utilizar a semana como eixo temporal no cálculo da tendência do CBL
df_tendencia_base = (
df_agg_bloco_semana
.withColumn(
"tempo_semana",
(F.unix_timestamp("semana") / 604800).cast("double")
)
)

In [0]:
# Calcula a tendência temporal do CBL para cada Bloco de 25 m a partir das agregações semanais
# O slope representa a inclinação da série: valores positivos indicam tendência crescente do indicador
df_degradacao = (
df_tendencia_base
.groupBy(
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m"
)
.agg(
# Quantidade de semanas distintas disponíveis para o cálculo da tendência
F.countDistinct("semana").alias("qtd_semanas"),
# Tendência temporal do CBL médio e do percentil 95
F.regr_slope("cbl_medio", "tempo_semana").alias("slope_cbl_medio"),
F.regr_slope("cbl_p95", "tempo_semana").alias("slope_cbl_p95"),
# Indicadores consolidados do comportamento do bloco no período analisado
F.avg("cbl_medio").alias("cbl_medio_periodo"),
F.max("cbl_max").alias("cbl_max_periodo"),
F.max("cbl_p95").alias("cbl_p95_max_periodo")
)
# Mantém somente blocos com histórico mínimo de quatro semanas para análise de tendência
.filter(F.col("qtd_semanas") >= 4)
)
# Exibe os blocos com maior tendência positiva do CBL P95
display(
df_degradacao
.orderBy(F.desc("slope_cbl_p95"))
.limit(20)
)

linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_semanas,slope_cbl_medio,slope_cbl_p95,cbl_medio_periodo,cbl_max_periodo,cbl_p95_max_periodo
LP002,LP002_518350_518375,518350,518375,4,0.0034302325581395928,0.00600775193798458,0.17875000000000002,0.25,0.25
LP002,LP002_842000_842025,842000,842025,4,0.001578947368421108,0.005526315789473807,0.1375,0.17,0.17
LP002,LP002_800525_800550,800525,800550,4,0.00538461538461527,0.00538461538461527,0.1225,0.14,0.14
SD001,SD001_65750_65775,65750,65775,4,0.00257142857142857,0.005142857142857144,0.1125,0.13,0.13
SD002,SD002_73625_73650,73625,73650,5,0.005074626865671546,0.005074626865671546,0.16199999999999998,0.23,0.23
LP002,LP002_712575_712600,712575,712600,6,0.005030927835051535,0.005030927835051535,0.1466666666666667,0.2,0.2
LP002,LP002_659075_659100,659075,659100,7,0.004749103942652576,0.004749103942652576,0.13714285714285715,0.21,0.21
LP002,LP002_296075_296100,296075,296100,16,0.002281357730045136,0.004472951471758238,0.16358333333333333,0.36,0.36
LP002,LP002_636675_636700,636675,636700,4,0.0049295774647887805,0.0043380281690141315,0.1375,0.18,0.18
LP002,LP002_742025_742050,742025,742050,4,0.004333333333333249,0.004333333333333249,0.1275,0.15,0.15


In [0]:
# Persiste os indicadores de tendência temporal do CBL por Bloco de 25 m na camada Gold
# Esta tabela será utilizada posteriormente para identificar os segmentos com maior tendência de crescimento do CBL
(
df_degradacao.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("mvp_vti.gold.agg_degradacao_bloco")
)